In [3]:
import os, pandas as pd
from sqlalchemy import create_engine

engine = create_engine(os.getenv("DATABASE_URL"))
df = pd.read_sql("SELECT * FROM raw_churn;", engine)

df.shape, df.dtypes


((7043, 21),
 customer_id           object
 gender                object
 senior_citizen         int64
 partner                 bool
 dependents              bool
 tenure                 int64
 phone_service           bool
 multiple_lines        object
 internet_service      object
 online_security       object
 online_backup         object
 device_protection     object
 tech_support          object
 streaming_tv          object
 streaming_movies      object
 contract              object
 paperless_billing       bool
 payment_method        object
 monthly_charges      float64
 total_charges        float64
 churn                   bool
 dtype: object)

In [8]:
from sklearn.model_selection import train_test_split
target = "churn"
X = df.drop(columns=[target])
y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10, stratify=y)

In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "bool"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
])


In [25]:
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix

clf.fit(X_train, y_train)

proba = clf.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, proba))
print("PR-AUC :", average_precision_score(y_test, proba))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

ROC-AUC: 0.8404402077036348
PR-AUC : 0.6484039784617499
[[792 243]
 [ 94 280]]
              precision    recall  f1-score   support

           0       0.89      0.77      0.82      1035
           1       0.54      0.75      0.62       374

    accuracy                           0.76      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.76      0.77      1409

